# Case-08.1:矩形單筋梁撓曲設計

依 [ROADMAP.md](../ROADMAP.md) 規劃,這是 **Case-08 系列**(鋼筋混凝土
入門到耐震設計配筋、不含 pushover)的起手式。

**這一步要解決的問題**:`case03_6`、`case03_7`、`case06_5` 一路以來,
柱跟梁的配筋都是**假設 rho=2%**,再檢查這個假設夠不夠用——從來沒有
真的從設計彎矩 Mu 反算出應該配多少鋼筋。Case-08 系列要把這個洞補起來,
從最基本的矩形單筋梁開始,08.8 會接回 `case03_7` 的 `design_loop`,
把固定 rho 換成這裡做出來的真實設計函式。

**定位**:這是 Design(配筋設計)案例,不含非線性分析/pushover——
產出的是符合規範強度設計法(《結構混凝土設計規範》第 21、22 章)的
配筋結果,可以直接被後續案例引用。

## 第 1 課:強度設計法公式

矩形單筋梁,拉力鋼筋降伏、混凝土壓應力用 Whitney 等效矩形應力塊簡化
(規範 22.2.2.4.1)。

**強度折減因數**(表 21.2.2):拉力控制斷面(εt ≥ 0.005)取 φ=0.9,
這裡先假設拉力控制,算完會反過來驗證 εt 是否真的 ≥0.005。

**係數 Rn 與需求鋼筋比**(標準強度設計法推導,由 Mu=φAsfy(d-a/2) 與
a=Asfy/(0.85f'cb) 消去 a 後解 As 的二次方程式得出):

$$R_n = \frac{M_u}{\phi b d^2}, \qquad
\rho = \frac{0.85f_c'}{f_y}\left(1-\sqrt{1-\frac{2R_n}{0.85f_c'}}\right)$$

**最小鋼筋比**(規範式 8.6.1.1,避免斷面開裂後立即脆性破壞):

$$\rho_{min} = \max\left(\frac{14}{f_y}, \frac{0.8\sqrt{f_c'}}{f_y}\right)$$

(以上兩式用規範原文傳統單位制:kgf/cm²、cm)

In [1]:
import math

def design_rebar(Mu_kNm, b_cm, h_cm, fc=280.0, fy=4200.0, cover=4.0,
                  stirrup_d=0.95, bar_d_guess=2.5, bar_table=None,
                  phi_target=0.9, beta1=0.85):
    '''
    矩形單筋梁撓曲配筋設計(強度設計法)。

    參數
    ----
    Mu_kNm : 設計彎矩需求 (kN-m)
    b_cm, h_cm : 梁寬、梁高 (cm)
    fc, fy : 混凝土抗壓強度、鋼筋降伏強度 (kgf/cm^2, 預設SD420)
    cover : 淨保護層 (cm)
    stirrup_d, bar_d_guess : 箍筋直徑、主筋預估直徑, 用來算有效深度d

    回傳
    ----
    dict, 含需求鋼筋比/As、選筋結果(含排筋間距檢核)、供給強度、使用率
    '''
    if bar_table is None:
        # (斷面積cm^2, 公稱直徑cm); 主筋一般從#5起跳, #4留給箍筋用
        bar_table = {
            "#5(D16)": (1.986, 1.59), "#6(D19)": (2.865, 1.91),
            "#7(D22)": (3.871, 2.22), "#8(D25)": (5.067, 2.54),
            "#9(D29)": (6.469, 2.87), "#10(D32)": (8.143, 3.22),
        }

    d = h_cm - cover - stirrup_d - bar_d_guess/2

    Mu_kgfcm = Mu_kNm*1e5/9.80665
    Rn = Mu_kgfcm/(phi_target*b_cm*d**2)
    disc = 1 - 2*Rn/(0.85*fc)
    if disc < 0:
        raise ValueError("斷面太小, 單筋設計無法滿足Mu, 需加大斷面或改用雙筋設計(見Case-08.2)")

    rho_req = (0.85*fc/fy)*(1-math.sqrt(disc))
    rho_min = max(14/fy, 0.8*math.sqrt(fc)/fy)
    rho_used = max(rho_req, rho_min)
    As_req = rho_used*b_cm*d

    a = As_req*fy/(0.85*fc*b_cm)
    c = a/beta1
    eps_t = 0.003*(d-c)/c
    if eps_t < 0.005:
        raise ValueError(f"eps_t={eps_t:.4f}<0.005, 非拉力控制斷面, "
                          "需要用過渡區phi內插或加大斷面")

    # ---- 選筋: 檢查單層排筋淨間距(規範最小值: db 或 2.5cm取大者) ----
    best = None
    for name, (Ab, db) in sorted(bar_table.items(), key=lambda kv: kv[1][0]):
        n = max(2, math.ceil(As_req/Ab))
        As_p = n*Ab
        clear_spacing = (b_cm - 2*cover - 2*stirrup_d - n*db)/(n-1) if n > 1 else None
        min_spacing = max(db, 2.5)
        if clear_spacing is None or clear_spacing < min_spacing:
            continue   # 這個尺寸單層排不下, 跳過
        over_ratio = As_p/As_req
        if best is None or over_ratio < best['over_ratio']:
            best = dict(bar_size=name, n_bars=n, As_provided=As_p,
                        over_ratio=over_ratio, clear_spacing=clear_spacing, bar_d=db)
    if best is None:
        raise ValueError("所有候選鋼筋尺寸單層都排不下, 需要雙層配筋或加大梁寬")

    a_p = best['As_provided']*fy/(0.85*fc*b_cm)
    Mn_p_kgfcm = best['As_provided']*fy*(d-a_p/2)
    phiMn_p = phi_target*Mn_p_kgfcm*9.80665e-5

    return dict(
        d=d, rho_req=rho_req, rho_min=rho_min, As_req=As_req,
        a=a, c=c, eps_t=eps_t, phi_used=phi_target,
        bar_size=best['bar_size'], n_bars=best['n_bars'], bar_d=best['bar_d'],
        As_provided=best['As_provided'], clear_spacing=best['clear_spacing'],
        phiMn_provided=phiMn_p, Mu_demand=Mu_kNm, utilization=Mu_kNm/phiMn_p,
    )

print("design_rebar() 定義完成")

design_rebar() 定義完成


## 第 2 課:設計案例

沿用之前對照過的案例(簡支梁 30×50cm, Wu=25kN/m, L=6m, 跨中
Mu=112.5kN-m),當作這個函式庫的標準測試案例——之後每次改函式,
都可以拿這組數字回歸測試。

In [2]:
Wu, L = 25.0, 6.0
b, h, cover = 30.0, 50.0, 4.0
Mu = Wu*L**2/8

r = design_rebar(Mu, b, h, cover=cover)

print(f"設計彎矩 Mu = {Mu:.1f} kN-m")
print(f"有效深度 d = {r['d']:.2f} cm")
print(f"需求鋼筋比 rho_req = {r['rho_req']:.5f}  (rho_min = {r['rho_min']:.5f})")
print(f"需求 As = {r['As_req']:.2f} cm^2")
print(f"拉力應變 eps_t = {r['eps_t']:.4f}  ({'拉力控制 phi=0.9 成立' if r['eps_t']>=0.005 else '需調整phi'})")
print(f"\n選筋: {r['n_bars']}-{r['bar_size']}, As供給 = {r['As_provided']:.2f} cm^2")
print(f"排筋淨間距 = {r['clear_spacing']:.2f} cm")
print(f"phiMn供給 = {r['phiMn_provided']:.2f} kN-m  (需求Mu = {Mu:.2f})")
print(f"使用率 = {r['utilization']:.1%}")

assert r['phiMn_provided'] >= Mu, "配筋強度不足, 設計錯誤"
assert r['eps_t'] >= 0.005, "非拉力控制斷面"
assert r['clear_spacing'] >= max(r['bar_d'], 2.5), "排筋間距不足"
print("\n[PASS] 三項檢核(強度足夠/拉力控制/排筋間距)全部通過")

設計彎矩 Mu = 112.5 kN-m
有效深度 d = 43.80 cm
需求鋼筋比 rho_req = 0.00554  (rho_min = 0.00333)
需求 As = 7.29 cm^2
拉力應變 eps_t = 0.0231  (拉力控制 phi=0.9 成立)

選筋: 2-#7(D22), As供給 = 7.74 cm^2
排筋淨間距 = 15.66 cm
phiMn供給 = 119.17 kN-m  (需求Mu = 112.50)
使用率 = 94.4%

[PASS] 三項檢核(強度足夠/拉力控制/排筋間距)全部通過


## 第 3 課:交叉驗證——用應變相容法反解 As

`design_rebar()` 用的是規範標準做法(Whitney 等效矩形應力塊 + 代數解),
這裡用**另一個獨立方法**——應變相容法(平面保持平面 + 力平衡,對中性軸
位置 c 做二分搜尋)反過來解:「要多少 As 才能剛好讓 φMn=Mu」,拿來驗證
公式解對不對。

理論上兩者應該完全一致(Whitney 應力塊在鋼筋降伏的前提下,本來就是
應變相容法在特定簡化假設下的封閉解,不是兩個獨立的物理模型)——
如果兩者對不上,代表 `design_rebar()` 或這裡的反解邏輯有 bug,不是
"合理的方法論差異"。

In [3]:
def find_As_by_strain_compat(Mu_kNm, b_cm, d_cm, fc=280.0, fy=4200.0, Es=2.0e6,
                              phi_target=0.9, beta1=0.85, eps_cu=0.003):
    '''給定Mu, 用應變相容法反解需要多少As(對中性軸c二分搜尋), 拿來跟公式解交叉驗證'''
    eps_y = fy/Es

    def phiMn_given_As(As):
        def force(c):
            eps_s = min(eps_cu*(d_cm-c)/c, eps_y)   # 假設拉力筋降伏(單筋梁常見情況)
            fs = Es*eps_s if eps_s < eps_y else fy
            a = beta1*c
            Cc = 0.85*fc*b_cm*a
            T = As*fs
            return Cc - T, a
        c_lo, c_hi = 0.01, d_cm
        for _ in range(100):
            c_mid = (c_lo+c_hi)/2
            diff, _ = force(c_mid)
            if diff > 0:
                c_hi = c_mid
            else:
                c_lo = c_mid
        _, a_final = force((c_lo+c_hi)/2)
        Mn = As*fy*(d_cm - a_final/2)
        return phi_target*Mn*9.80665e-5

    As_lo, As_hi = 0.1, 50.0
    for _ in range(60):
        As_mid = (As_lo+As_hi)/2
        if phiMn_given_As(As_mid) < Mu_kNm:
            As_lo = As_mid
        else:
            As_hi = As_mid
    return (As_lo+As_hi)/2

As_cross_check = find_As_by_strain_compat(Mu, b, r['d'])
diff_pct = abs(r['As_req']-As_cross_check)/r['As_req']*100

print(f"公式解(design_rebar)   As_req = {r['As_req']:.4f} cm^2")
print(f"應變相容法反解          As_req = {As_cross_check:.4f} cm^2")
print(f"差異 = {diff_pct:.3f}%")

assert diff_pct < 0.1, "兩個方法差異過大, 檢查design_rebar()或反解邏輯是否有bug"
print("\n[PASS] 兩個獨立方法完全一致, design_rebar()公式解無誤")

公式解(design_rebar)   As_req = 7.2853 cm^2
應變相容法反解          As_req = 7.2853 cm^2
差異 = 0.000%

[PASS] 兩個獨立方法完全一致, design_rebar()公式解無誤


## 小結

- `design_rebar()` 正式收斂完成,含:規範強度公式、最小鋼筋比、拉力
  控制檢核、選筋(含排筋間距檢核)——之後 Case-08.2 起會延伸雙筋梁、
  T形梁,08.8 會接回 `case03_7` 取代固定的 `rho=0.02` 假設。
- 這一步的交叉驗證跟之前 fiber section vs 應變相容法的驗證**性質不同**:
  這裡兩個方法理論上該完全相等(同一個力學模型的兩種解法),對不上就是
  bug;fiber vs 應變相容法那次是兩個**不同材料模型假設**,允許有合理
  差異。兩種"疊圖對不對得起來"的判斷標準不能混為一談。